# 第12周 · 作业（预计约7小时）

| 部分 | 内容 | 建议时间 |
| --- | --- | --- |
| A | 字符级语言模型的消融实验（二选一：位置编码有/无，或上下文长度 32 vs 128），3 个种子，写实验记录 | 4.5 小时 |
| B | 根据自审意见修订 Capstone 实验设计文档（第二版） | 2 小时 |
| C | 代码讲解段落 + 反思 | 30 分钟 |

**约定**：可以使用 AI 编程助手，但 C 部分一定要写**代码讲解段落**（用自己的话说明 `run_once` 每一段在做什么、关键张量的 shape）；写完再随机挑一段，给自己讲一遍并做一处小修改、先预测结果再运行。

模型与数据流水线照讲义（结构照 karpathy/ng-video-lecture 的 gpt.py，MIT 许可），下面直接给出；你的工作是**设计并执行一个干净的实验**。全程用讲义里的 CPU 配置，每次训练不到 1 分钟，普通笔记本就能做完全部 6 次训练。

In [ ]:
# 运行这一格来安装课程工具包（每次在 Colab 打开 notebook 后运行一次）
import importlib.util, subprocess, sys
if importlib.util.find_spec("zero2researcher") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/ReaganGen/Zero2Researcher"], check=True)
print("课程工具包已就绪")

In [ ]:
import math
import time

import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.nn import functional as F
from zero2researcher import data, experiments
from zero2researcher.checks import week12 as ck

# 课程统一的 device 选择：有 CUDA 用 CUDA（Colab GPU），Apple 芯片用 MPS，否则 CPU
device = torch.device("cuda" if torch.cuda.is_available()
                      else "mps" if torch.backends.mps.is_available() else "cpu")
print("torch", torch.__version__, "| device:", device)

## A · 消融实验：位置编码，还是上下文长度？

### 给定的流水线

数据（Tiny Shakespeare，前 90% 训练、后 10% 验证）、`get_batch`、`estimate_loss` 与模型。`GPTLanguageModel` 有两个旋钮：`use_pos`（是否加位置 embedding）和 `block_size`（上下文长度）。

> 对照视频时注意：`Head` 里的缩放用的是 $\sqrt{\text{head\_size}}$。Karpathy 视频中现场写的 gpt.py 是 `C**-0.5`（`C = n_embd`，所有头的总维度）；仓库在 2023-01-25 的提交 988848a（"scale by head size"）中改成了 head_size。本课程采用修正后的写法，与 Vaswani et al. 2017 §3.2.1 的 $\sqrt{d_k}$ 一致（讲义 S2 有详细说明）。

In [ ]:
text = data.load_shakespeare()
chars = sorted(set(text))
vocab_size = len(chars)
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: "".join([itos[i] for i in l])

data_ids = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data_ids))
train_data, val_data = data_ids[:n], data_ids[n:]


def get_batch(data, block_size, batch_size):
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i + block_size] for i in ix])               # (B, T)
    y = torch.stack([data[i + 1:i + block_size + 1] for i in ix])       # (B, T)
    return x.to(device), y.to(device)


@torch.no_grad()
def estimate_loss(model, data, block_size, batch_size=32, eval_iters=100):
    """在 data 上取 eval_iters 个随机 batch，返回平均 loss"""
    model.eval()
    losses = [model(*get_batch(data, block_size, batch_size))[1].item() for _ in range(eval_iters)]
    model.train()
    return float(np.mean(losses))

In [ ]:
class Head(nn.Module):
    """一个因果自注意力头"""

    def __init__(self, n_embd, head_size, block_size, dropout=0.0):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer("tril", torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)                                                  # (B, T, hs)
        q = self.query(x)                                                # (B, T, hs)
        wei = q @ k.transpose(-2, -1) * k.shape[-1] ** -0.5              # (B, T, T)：除以 √head_size（视频版是 C**-0.5）
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float("-inf"))     # 因果掩码
        wei = self.dropout(F.softmax(wei, dim=-1))                       # (B, T, T)
        return wei @ self.value(x)                                       # (B, T, hs)


class MultiHeadAttention(nn.Module):
    def __init__(self, n_embd, num_heads, block_size, dropout=0.0):
        super().__init__()
        head_size = n_embd // num_heads
        self.heads = nn.ModuleList([Head(n_embd, head_size, block_size, dropout) for _ in range(num_heads)])
        self.proj = nn.Linear(head_size * num_heads, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)             # (B, T, C)
        return self.dropout(self.proj(out))                              # (B, T, C)


class FeedForward(nn.Module):
    def __init__(self, n_embd, dropout=0.0):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(n_embd, 4 * n_embd), nn.ReLU(),
                                 nn.Linear(4 * n_embd, n_embd), nn.Dropout(dropout))

    def forward(self, x):
        return self.net(x)


class Block(nn.Module):
    def __init__(self, n_embd, n_head, block_size, dropout=0.0):
        super().__init__()
        self.sa = MultiHeadAttention(n_embd, n_head, block_size, dropout)
        self.ffwd = FeedForward(n_embd, dropout)
        self.ln1, self.ln2 = nn.LayerNorm(n_embd), nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))                                     # (B, T, C)
        return x + self.ffwd(self.ln2(x))                                # (B, T, C)


class GPTLanguageModel(nn.Module):
    """字符级 GPT（结构照 karpathy/ng-video-lecture 的 gpt.py，MIT）；use_pos=False 去掉位置 embedding"""

    def __init__(self, vocab_size, n_embd=64, n_head=2, n_layer=2, block_size=64, dropout=0.0, use_pos=True):
        super().__init__()
        self.block_size, self.use_pos = block_size, use_pos
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head, block_size, dropout) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
        if isinstance(module, nn.Linear) and module.bias is not None:
            nn.init.zeros_(module.bias)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        x = self.token_embedding_table(idx)                              # (B, T, C)
        if self.use_pos:
            x = x + self.position_embedding_table(torch.arange(T, device=idx.device))   # + (T, C)
        logits = self.lm_head(self.ln_f(self.blocks(x)))                 # (B, T, vocab_size)
        if targets is None:
            return logits, None
        B, T, C = logits.shape
        return logits, F.cross_entropy(logits.view(B * T, C), targets.view(B * T))

### 你的实验

**二选一**（都做更好，解答版两个都做了）：

| | 选项 1：位置编码 | 选项 2：上下文长度 |
| --- | --- | --- |
| 研究问题 | 去掉位置 embedding，字符级 GPT 的验证 loss 会变差多少？ | 上下文从 32 拉长到 128，验证 loss 能降多少？ |
| 自变量 | `use_pos` ∈ {True, False}（`block_size = 64`） | `block_size` ∈ {32, 128}（`use_pos = True`） |
| 控制变量 | 其余全部相同：2 层、2 头、`n_embd = 64`、`batch_size = 32`、AdamW 1e-3、**1000 步**、`eval_iters = 100` | 同左 |
| 种子 | 0、1、2（每次训练前 `experiments.set_seed(seed)`） | 同左 |
| 报告 | 每个条件 3 个验证 loss 的均值 ± 标准差（`experiments.format_result`）+ 误差条图 | 同左 |

**先写预测再跑**：在下面的实验记录里先填"假设"一栏，再运行。第 11 周的序列反转里，没有位置编码的编码器卡在了众数天花板——想一想：GPT 有**因果掩码**，情况会一样吗？

第一步，写 `run_once`：建模型 → 训练 `max_iters` 步（第 8 周的五步）→ 返回验证集 loss。

In [ ]:
def run_once(seed, use_pos=True, block_size=64, max_iters=1000, batch_size=32, lr=1e-3):
    """训练一个讲义 CPU 配置的 GPT，返回最终验证集 loss（float）"""
    experiments.set_seed(seed)
    model = GPTLanguageModel(vocab_size, n_embd=64, n_head=2, n_layer=2,
                             block_size=block_size, use_pos=use_pos).to(device)
    # 在这里写你的代码
    raise NotImplementedError


t0 = time.perf_counter()
print(f"冒烟测试（50 步）：val loss {run_once(0, max_iters=50):.3f}   {time.perf_counter() - t0:.1f} 秒")

**选项 1 · 位置编码**：结果存成 `{"with PE": [三个种子的 val loss], "no PE": [...]}`。

In [ ]:
results_pe = {}
t0 = time.perf_counter()
# 在这里写你的代码
raise NotImplementedError
for label, vals in results_pe.items():
    print(f"{label:8s} {experiments.format_result(vals)}   各种子 {np.round(vals, 4).tolist()}")
print(f"用时 {time.perf_counter() - t0:.0f} 秒")
ck.ablation_results(results_pe)

**选项 2 · 上下文长度**：结果存成 `{"block 32": [...], "block 128": [...]}`。（`block_size = 128` 每次训练约 40 秒，比 32 慢三四倍。）

In [ ]:
results_ctx = {}
t0 = time.perf_counter()
# 在这里写你的代码
raise NotImplementedError
for label, vals in results_ctx.items():
    print(f"{label:9s} {experiments.format_result(vals)}   各种子 {np.round(vals, 4).tolist()}")
print(f"用时 {time.perf_counter() - t0:.0f} 秒")
ck.ablation_results(results_ctx)

画误差条图（每个条件一个点、误差条 = 标准差），并加两条水平参考线：bigram 基线 2.48 与均匀基线 $\ln 65 = 4.17$ 中你觉得有用的那条。标题写结论。

In [ ]:
# 在这里写你的代码
raise NotImplementedError


## 实验记录 #N · 日期：____

（选了哪个选项就写哪个；两个都做了就写两条。）

- 研究问题：
- 假设（预期结果，以及什么结果会推翻它）——**运行前填写**：
- 自变量（这次只改变什么）：
- 控制变量（哪些保持不变；有没有被"顺带"改变的东西？）：
- 基线：
- 数据与划分方式（如何避免泄漏）：
- 随机种子：
- 结果（数值：均值 ± 标准差；图：见上）：
- 解释：
- 局限：
- 下一步：

## B · Capstone 实验设计文档（第二版）

把第 10 周的一页提案，按讲义 S4 自审中发现的问题扩写成完整的实验设计（模板见学习指南 7.3 节）。先填"自审意见与修改"表，再改正文——每一条意见都要有回应（接受并修改，或说明为什么不改）。

### 自审意见与修改

| # | 意见（来自自审、学习伙伴或 AI 审稿） | 我的回应：接受 / 不接受 | 改了哪里 |
| --- | --- | --- | --- |
| 1 | | | |
| 2 | | | |
| 3 | | | |

### 实验设计

1. 研究问题与假设
2. 数据：来源、预处理、划分方式（说明如何避免泄漏）
3. 模型：待测模型与基线阶梯
4. 自变量 / 控制变量 / 需单独调优的干扰变量
5. 评价指标（以及为什么选它）
6. 种子数与统计方法
7. 消融计划：去掉或替换哪个组件
8. 最大风险（最可能让项目失败的假设）及最先验证的方法
9. 时间表：第16、18、19、20周各完成什么

## C · 代码讲解段落 + 反思

**代码讲解**（必做）：用 6–10 句话讲解你写的 `run_once`，以及给定的 `GPTLanguageModel.forward`：每一步在做什么、`idx`、`x`、`logits`、`targets` 各是什么 shape、为什么 `cross_entropy` 之前要 `view(B * T, C)`。如果用了 AI 助手，写明它帮你做了什么、你怎么验证的。

**反思**（2–3 句话）：你的实验结果和"运行前的假设"一致吗？如果不一致，是哪一步推理出了问题？

（在这里写）